## Parte 1: Configuração e criação do banco de dados

Importação das bibliotecas

In [3]:
import sqlite3
import pandas as pd

Criação da conexão

In [4]:
conn = sqlite3.connect("empresa.db")
cursor = conn.cursor()
print("Conexão com o banco de dados realizada com sucesso!")

Conexão com o banco de dados realizada com sucesso!


Criação da tabela `funcionarios`

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS funcionarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cargo TEXT NOT NULL,
    salario REAL,
    data_contratacao TEXT NOT NULL
);
""")
conn.commit()
print("Tabela funcionarios criada com sucesso!")

Tabela funcionarios criada com sucesso!


Verificação da estrutura da tabela

In [6]:
cursor.execute("PRAGMA table_info(funcionarios);")
estrutura = cursor.fetchall()
pd.DataFrame(estrutura, columns=["cid", "nome", "tipo", "notnull", "valor_padrao", "chave_primaria"])

,cid,nome,tipo,notnull,valor_padrao,chave_primaria
0,0,id,INTEGER,0,None,1
1,1,nome,TEXT,1,None,0
2,2,cargo,TEXT,1,None,0
3,3,salario,REAL,0,None,0
4,4,data_contratacao,TEXT,1,None,0


## Parte 2: Manipulação de dados (DML)

Inserção de registros

In [7]:
funcionarios = [
    ("Joao", "Analista", 4800.00, "2024-02-15"),
    ("Maria", "Gerente", 7500.00, "2023-08-01"),
    ("Carlos", "Desenvolvedor", 6200.00, "2024-01-10"),
    ("Ana", "Gerente", 8300.00, "2022-11-20"),
    ("Beatriz", "Assistente", 3500.00, "2024-03-05")
]
cursor.executemany("""
    INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
    VALUES (?, ?, ?, ?)
""", funcionarios)
conn.commit()
print(f"{cursor.rowcount} registros inseridos.")

5 registros inseridos.


Consultas, filtros, atualização, exclusão e contagem

In [8]:
def consultar_dataframe(sql, parametros=()):
    return pd.read_sql_query(sql, conn, params=parametros)

todos = consultar_dataframe("SELECT * FROM funcionarios;")
acima_de_5000 = consultar_dataframe("SELECT * FROM funcionarios WHERE salario > ?;", (5000,))
gerentes = consultar_dataframe("SELECT * FROM funcionarios WHERE cargo = ?;", ("Gerente",))

display(todos)
display(acima_de_5000)
display(gerentes)

,id,nome,cargo,salario,data_contratacao
0,1,Joao,Analista,4800.0,2024-02-15
1,2,Maria,Gerente,7500.0,2023-08-01
2,3,Carlos,Desenvolvedor,6200.0,2024-01-10
3,4,Ana,Gerente,8300.0,2022-11-20
4,5,Beatriz,Assistente,3500.0,2024-03-05


,id,nome,cargo,salario,data_contratacao
0,2,Maria,Gerente,7500.0,2023-08-01
1,3,Carlos,Desenvolvedor,6200.0,2024-01-10
2,4,Ana,Gerente,8300.0,2022-11-20


,id,nome,cargo,salario,data_contratacao
0,2,Maria,Gerente,7500.0,2023-08-01
1,4,Ana,Gerente,8300.0,2022-11-20


In [9]:
cursor.execute("UPDATE funcionarios SET salario = salario * 1.10 WHERE nome = ?;", ("Joao",))
conn.commit()

cursor.execute("DELETE FROM funcionarios WHERE nome = ?;", ("Beatriz",))
conn.commit()

quantidade = cursor.execute("SELECT COUNT(*) FROM funcionarios;").fetchone()[0]
print(f"Total de funcionarios: {quantidade}")

Total de funcionarios: 4


Agregação, agrupamento e ordenação

In [10]:
resumo_salarios = consultar_dataframe("""
    SELECT
        SUM(salario) AS soma_total,
        ROUND(AVG(salario), 2) AS salario_medio,
        MAX(salario) AS salario_maximo,
        MIN(salario) AS salario_minimo
    FROM funcionarios;
""")
display(resumo_salarios)

media_por_cargo = consultar_dataframe("""
    SELECT cargo, ROUND(AVG(salario), 2) AS salario_medio
    FROM funcionarios
    GROUP BY cargo;
""")
quantidade_por_cargo = consultar_dataframe("""
    SELECT cargo, COUNT(*) AS quantidade
    FROM funcionarios
    GROUP BY cargo;
""")
display(media_por_cargo)
display(quantidade_por_cargo)

,soma_total,salario_medio,salario_maximo,salario_minimo
0,27280.0,6820.0,8300.0,5280.0


,cargo,salario_medio
0,Analista,5280.0
1,Desenvolvedor,6200.0
2,Gerente,7900.0


,cargo,quantidade
0,Analista,1
1,Desenvolvedor,1
2,Gerente,2


In [11]:
salarios_decrescente = consultar_dataframe("SELECT * FROM funcionarios ORDER BY salario DESC;")
nomes_alfabetica = consultar_dataframe("SELECT * FROM funcionarios ORDER BY nome ASC;")
display(salarios_decrescente)
display(nomes_alfabetica)

,id,nome,cargo,salario,data_contratacao
0,4,Ana,Gerente,8300.0,2022-11-20
1,2,Maria,Gerente,7500.0,2023-08-01
2,3,Carlos,Desenvolvedor,6200.0,2024-01-10
3,1,Joao,Analista,5280.0,2024-02-15


,id,nome,cargo,salario,data_contratacao
0,4,Ana,Gerente,8300.0,2022-11-20
1,3,Carlos,Desenvolvedor,6200.0,2024-01-10
2,1,Joao,Analista,5280.0,2024-02-15
3,2,Maria,Gerente,7500.0,2023-08-01


Transações, `ROLLBACK` e `COMMIT`

Simulação de erro e `ROLLBACK`

In [12]:
try:
    conn.execute("BEGIN;")
    cursor.execute("""
        INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
        VALUES (?, ?, ?, ?)
    """, ("Funcionario Rollback", "Estagiario", 2500.00, "2025-01-15"))
    raise ValueError("Erro proposital para testar o rollback")
except ValueError as erro:
    conn.rollback()
    print(f"{erro}. Transação desfeita com sucesso.")

verificacao_rollback = consultar_dataframe("SELECT * FROM funcionarios WHERE nome = ?;", ("Funcionario Rollback",))
display(verificacao_rollback)

Erro proposital para testar o rollback. Transação desfeita com sucesso.


,id,nome,cargo,salario,data_contratacao


Transação confirmada com `COMMIT`

In [13]:
conn.execute("BEGIN;")
cursor.execute("""
    INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
    VALUES (?, ?, ?, ?)
""", ("Funcionario Commit", "Analista", 5100.00, "2025-02-03"))
conn.commit()
print("Transação confirmada com sucesso.")

verificacao_commit = consultar_dataframe("SELECT * FROM funcionarios WHERE nome = ?;", ("Funcionario Commit",))
display(verificacao_commit)

Transação confirmada com sucesso.


,id,nome,cargo,salario,data_contratacao
0,6,Funcionario Commit,Analista,5100.0,2025-02-03


## Parte 5: Joins e Múltiplas Tabelas

Tabela chamada departamentos

In [14]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS departamentos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_departamento TEXT NOT NULL,
    localizacao TEXT
);
""")
conn.commit()

Inserção de registros na tabela `departamentos`

Inserção de departamentos

In [15]:
departamentos = [
    ("Financeiro", "São Paulo"),
    ("TI", "Rio de Janeiro"),
    ("Recursos Humanos", "Belo Horizonte")
]

cursor.executemany("""
    INSERT INTO departamentos (nome_departamento, localizacao)
    VALUES (?, ?)
""", departamentos)
conn.commit()
print(f"{cursor.rowcount} departamentos inseridos.")

consulta_departamentos = consultar_dataframe("SELECT * FROM departamentos;")
display(consulta_departamentos)


3 departamentos inseridos.


,id,nome_departamento,localizacao
0,1,Financeiro,São Paulo
1,2,TI,Rio de Janeiro
2,3,Recursos Humanos,Belo Horizonte


Adição da coluna `departamento_id` e chave estrangeira

In [16]:
cursor.execute("ALTER TABLE funcionarios ADD COLUMN departamento_id INTEGER;")
conn.commit()

cursor.execute("""
    UPDATE funcionarios
    SET departamento_id = CASE
        WHEN nome = 'Joao' THEN 1
        WHEN nome = 'Maria' THEN 2
        WHEN nome = 'Carlos' THEN 2
        WHEN nome = 'Ana' THEN 3
        WHEN nome = 'Funcionario Commit' THEN 1
        WHEN nome = 'Funcionario Rollback' THEN 3
    END
""")
conn.commit()

print("Coluna departamento_id adicionada e funcionários vinculados aos departamentos.")

consulta_funcionarios = consultar_dataframe("SELECT id, nome, departamento_id FROM funcionarios;")
display(consulta_funcionarios)


Coluna departamento_id adicionada e funcionários vinculados aos departamentos.


,id,nome,departamento_id
0,1,Joao,1
1,2,Maria,2
2,3,Carlos,2
3,4,Ana,3
4,6,Funcionario Commit,1


INNER JOIN entre `funcionarios` e `departamentos`

In [17]:
inner_join = consultar_dataframe("""
    SELECT f.nome, f.cargo, d.nome_departamento, d.localizacao
    FROM funcionarios AS f
    INNER JOIN departamentos AS d ON f.departamento_id = d.id;
""")
display(inner_join)


,nome,cargo,nome_departamento,localizacao
0,Joao,Analista,Financeiro,São Paulo
1,Maria,Gerente,TI,Rio de Janeiro
2,Carlos,Desenvolvedor,TI,Rio de Janeiro
3,Ana,Gerente,Recursos Humanos,Belo Horizonte
4,Funcionario Commit,Analista,Financeiro,São Paulo


Criação das tabelas `projetos` e `funcionarios_projetos`

In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS projetos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_projeto TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS funcionarios_projetos (
    funcionario_id INTEGER,
    projeto_id INTEGER,
    FOREIGN KEY (funcionario_id) REFERENCES funcionarios(id),
    FOREIGN KEY (projeto_id) REFERENCES projetos(id)
);
""")
conn.commit()

projetos = [("Sistema de Gestão",), ("CRM",), ("App de Vendas",)]
cursor.executemany("INSERT INTO projetos (nome_projeto) VALUES (?)", projetos)

relacoes = [
    (1, 1),
    (1, 2),
    (2, 1),
    (3, 3)
]
cursor.executemany("INSERT INTO funcionarios_projetos (funcionario_id, projeto_id) VALUES (?, ?)", relacoes)
conn.commit()

print("Tabelas de projetos e relacionamento criadas com sucesso.")

consulta_relacoes = consultar_dataframe("SELECT * FROM funcionarios_projetos;")
display(consulta_relacoes)


Tabelas de projetos e relacionamento criadas com sucesso.


,funcionario_id,projeto_id
0,1,1
1,1,2
2,2,1
3,3,3


LEFT JOIN para listar todos os funcionários e seus projetos

In [20]:
left_join = consultar_dataframe("""
    SELECT f.nome, p.nome_projeto
    FROM funcionarios AS f
    LEFT JOIN funcionarios_projetos AS fp ON f.id = fp.funcionario_id
    LEFT JOIN projetos AS p ON fp.projeto_id = p.id;
""")
display(left_join)


,nome,nome_projeto
0,Joao,Sistema de Gestão
1,Joao,CRM
2,Maria,Sistema de Gestão
3,Carlos,App de Vendas
4,Ana,NaN
5,Funcionario Commit,NaN


## Parte 6: DCL (Controle de Dados)

O DCL (Data Control Language) são comando SQL responsáveis pelo controle de acesso e permissões sobre os dados e objetos do banco. Ele define quem podem fazer no banco, garantindo segurança e organização do ambiente.

Diferente do DDL, DML e DQL:
- DDL: define a estrutura do banco, como criar, alterar e excluir tabelas. Exemplo: CREATE TABLE, ALTER TABLE.
- DML: manipula os dados dentro das tabelas. Exemplo: INSERT, UPDATE, DELETE.
- DQL: consulta os dados existentes. Exemplo: SELECT.

Em resumo, DCL é essencial em ambientes em que vários usuários acessam o mesmo banco, pois define as regras de permissão e não deixa ocorre um erro sem quere por parte de outro.